# Support Ticket Classification & Prioritization System

## Project Overview

Customer support teams receive hundreds or thousands of tickets daily. Manually reading each ticket, deciding which department it belongs to, and judging how urgently it must be handled is time-consuming and error-prone.

This project builds an **end-to-end NLP machine learning pipeline** that automatically:

1. **Classifies** the ticket into one of four categories:  
   `Billing | Technical Issue | Account | General Query`
2. **Predicts** the priority level:  
   `High | Medium | Low`

### Business Value
- Routes tickets to the correct team instantly (no manual triage).
- Flags high-priority issues so they are handled first.
- Reduces average resolution time and improves customer satisfaction.
- Scales to any ticket volume without additional headcount.

---

## 1. Import Libraries

In [ ]:
import os
import sys
import re
import string
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Download NLTK resources
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)

print('All libraries imported successfully!')

## 2. Load the Dataset

In [ ]:
# Resolve path relative to this notebook
NOTEBOOK_DIR = os.path.abspath('')
BASE_DIR = os.path.dirname(NOTEBOOK_DIR)
DATA_PATH = os.path.join(BASE_DIR, 'data', 'support_tickets.csv')

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# Basic info
print('\nDataset Info:')
df.info()

print('\nMissing values:')
print(df.isnull().sum())

print('\nCategory counts:')
print(df['category'].value_counts())

print('\nPriority counts:')
print(df['priority'].value_counts())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category distribution
df['category'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Ticket Category Distribution', fontsize=14)
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Priority distribution
df['priority'].value_counts().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90
)
axes[1].set_title('Ticket Priority Distribution', fontsize=14)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Category vs Priority heatmap
cross = pd.crosstab(df['category'], df['priority'])
plt.figure(figsize=(8, 4))
sns.heatmap(cross, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Category vs Priority Cross-tabulation')
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

We apply the following steps to each ticket:
1. **Lowercase** – normalise case
2. **Remove punctuation** – reduce noise
3. **Remove digits** – rarely informative
4. **Tokenise** – split into words
5. **Remove stopwords** – drop common words (the, is, …)
6. **Lemmatise** – reduce words to their base form

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """Preprocess raw ticket text."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 3. Remove digits
    text = re.sub(r'\d+', '', text)
    # Clean extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # 4. Tokenise
    tokens = word_tokenize(text)
    # 5 & 6. Stopword removal + lemmatisation
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 1
    ]
    return ' '.join(tokens)

df['cleaned_text'] = df['ticket_text'].astype(str).apply(clean_text)

# Preview
df[['ticket_text', 'cleaned_text']].head(5)

## 5. TF-IDF Vectorisation

**TF-IDF** (Term Frequency – Inverse Document Frequency) converts each cleaned ticket into a numerical vector that machine learning models can consume.

- **TF** measures how often a word appears in a ticket.
- **IDF** down-weights words that appear in many tickets (less informative).

In [ ]:
# max_features=5000 keeps the vocabulary manageable while retaining the most
# informative tokens; ngram_range=(1,2) captures single words and two-word
# phrases (e.g. 'password reset', 'billing issue') that carry strong signal.
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = tfidf.fit_transform(df['cleaned_text'])

y_category = df['category']
y_priority = df['priority']

print(f'TF-IDF matrix shape: {X.shape}')
print(f'Vocabulary size    : {len(tfidf.vocabulary_)}')

## 6. Train/Test Split

In [ ]:
X_train, X_test, yc_train, yc_test = train_test_split(
    X, y_category, test_size=0.2, random_state=42, stratify=y_category
)
_, _, yp_train, yp_test = train_test_split(
    X, y_priority, test_size=0.2, random_state=42, stratify=y_priority
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

## 7. Model Training & Evaluation

We train three classifiers:
- **Logistic Regression** – strong linear baseline
- **Naive Bayes (Multinomial)** – classic text classification model
- **Random Forest** – ensemble of decision trees

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    """Fit a model and return metric dict."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    metrics = {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, average='weighted', zero_division=0),
        'Recall'   : recall_score(y_te, y_pred, average='weighted', zero_division=0),
        'F1'       : f1_score(y_te, y_pred, average='weighted', zero_division=0),
    }

    print(f'\n── {name} ──')
    for k, v in metrics.items():
        if k != 'Model':
            print(f'  {k:<10}: {v:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_te, y_pred, zero_division=0))

    return metrics, model, confusion_matrix(y_te, y_pred), model.classes_, y_pred


MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes'        : MultinomialNB(),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
}

### 7a. Ticket Category Classifier

In [ ]:
print('=' * 55)
print('TICKET CATEGORY CLASSIFIER')
print('=' * 55)

cat_results = {}
for name, model in MODELS.items():
    metrics, fitted_model, cm, classes, y_pred = evaluate(
        name, model, X_train, X_test, yc_train, yc_test
    )
    cat_results[name] = {
        **metrics,
        'model': fitted_model,
        'cm': cm,
        'classes': classes,
        'y_pred': y_pred,
    }

### 7b. Priority Prediction Model

In [ ]:
print('=' * 55)
print('PRIORITY PREDICTION MODEL')
print('=' * 55)

# Fresh model instances to avoid any state leakage
pri_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes'        : MultinomialNB(),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
}

pri_results = {}
for name, model in pri_models.items():
    metrics, fitted_model, cm, classes, y_pred = evaluate(
        name, model, X_train, X_test, yp_train, yp_test
    )
    pri_results[name] = {
        **metrics,
        'model': fitted_model,
        'cm': cm,
        'classes': classes,
        'y_pred': y_pred,
    }

## 8. Results Visualisation

In [ ]:
def plot_confusion_matrix(cm, classes, title):
    """Plot a labelled confusion matrix heatmap."""
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(title, fontsize=13)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()


def plot_model_comparison(results, task_label):
    """Bar chart comparing Accuracy, Precision, Recall, F1 across models."""
    names   = list(results.keys())
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
    x = np.arange(len(names))
    width = 0.2

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, metric in enumerate(metrics):
        values = [results[n][metric] for n in names]
        ax.bar(x + i * width, values, width, label=metric)

    ax.set_title(f'Model Performance – {task_label}', fontsize=13)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(names, rotation=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Identify best models
best_cat_name = max(cat_results, key=lambda k: cat_results[k]['F1'])
best_pri_name = max(pri_results, key=lambda k: pri_results[k]['F1'])

print(f'Best Category Model: {best_cat_name}  (F1={cat_results[best_cat_name]["F1"]:.4f})')
print(f'Best Priority Model: {best_pri_name}  (F1={pri_results[best_pri_name]["F1"]:.4f})')

In [ ]:
# Confusion matrices for best models
plot_confusion_matrix(
    cat_results[best_cat_name]['cm'],
    cat_results[best_cat_name]['classes'],
    f'Category Classifier – {best_cat_name}'
)

plot_confusion_matrix(
    pri_results[best_pri_name]['cm'],
    pri_results[best_pri_name]['classes'],
    f'Priority Classifier – {best_pri_name}'
)

In [ ]:
# Performance comparison across all models
plot_model_comparison(cat_results, 'Ticket Category')
plot_model_comparison(pri_results, 'Ticket Priority')

## 9. Sample Prediction

Now let's use the best models to predict the category and priority of new tickets.

In [ ]:
best_cat_model = cat_results[best_cat_name]['model']
best_pri_model = pri_results[best_pri_name]['model']

def predict_ticket(ticket_text: str) -> dict:
    """Predict category and priority for a raw ticket string."""
    cleaned   = clean_text(ticket_text)
    features  = tfidf.transform([cleaned])
    category  = best_cat_model.predict(features)[0]
    priority  = best_pri_model.predict(features)[0]
    return {'ticket': ticket_text, 'category': category, 'priority': priority}


# ── Demo from the problem statement ───────────────────────────────────────────
ticket = 'I cannot login to my account and password reset is not working.'
result = predict_ticket(ticket)

print('Input Ticket:')
print(f'  "{result["ticket"]}"')
print()
print(f'Category → {result["category"]}')
print(f'Priority → {result["priority"]}')

In [ ]:
# Additional demo tickets
demo_tickets = [
    'I was charged twice for my monthly subscription.',
    'The application keeps crashing whenever I open settings.',
    'How do I upgrade my subscription plan?',
    'My account has been locked and I need urgent help.',
]

print(f'{"Ticket":<55} {"Category":<20} {"Priority"}')
print('─' * 90)
for t in demo_tickets:
    r = predict_ticket(t)
    print(f'{t[:54]:<55} {r["category"]:<20} {r["priority"]}')

## 10. Save Models

In [ ]:
MODEL_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

with open(os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)

with open(os.path.join(MODEL_DIR, 'category_model.pkl'), 'wb') as f:
    pickle.dump(best_cat_model, f)

with open(os.path.join(MODEL_DIR, 'priority_model.pkl'), 'wb') as f:
    pickle.dump(best_pri_model, f)

print('Models saved to models/')

---

## Summary

| Task | Best Model | F1 Score |
|------|-----------|----------|
| Ticket Category | See output above | – |
| Ticket Priority | See output above | – |

### How This System Helps Companies

| Benefit | Description |
|---------|-------------|
| **Faster Routing** | Tickets land in the right team inbox automatically. |
| **Priority Handling** | High-priority issues are surfaced immediately. |
| **Reduced Manual Work** | Support staff spend time solving, not sorting. |
| **Scalability** | Handles thousands of tickets per second. |
| **Consistency** | Removes human bias from categorisation decisions. |